# Ch.4 演習ノートブック：機械学習モデル構築

| 項目 | 内容 |
|------|------|
| **使用データ** | ワインデータ（`load_wine()`）178件・13特徴量・3品種 |
| **作るもの** | 正解率・混同行列・特徴量重要度の「評価3点セット」 |
| **実務での対応物** | モデル評価レポート・経営層への性能説明資料 |
| **時間の目安** | 演習 40分（問3まで完了で十分） |

## このノートブックの 2 大重要ルール

> 🔑 **ルール①（データリーク防止）：**  
> `scaler.fit` は**訓練データのみ**。テストデータには `transform` だけ使う。

> 🔑 **ルール②（正しい評価）：**  
> 正解率だけで評価しない。必ず**混同行列とセット**で確認する。

> ✅ **問3まで完了すれば十分です。**

---
## 🤖 GitHub Copilot の使い方（このコースでの活用ルール）

### JupyterLab で Copilot を使う 2 つの方法

| 方法 | 手順 |
|------|------|
| **コメントから補完** | コードセルに `# 〇〇するコードを書いて` と書いて **Tab** キー |
| **チャットで質問** | 右サイドバーの Copilot Chat に日本語で質問する |

### このコースでの活用ルール

| タイミング | ルール |
|-----------|--------|
| 座学中 | **禁止** ― まず自分の頭で概念を理解する |
| 演習 問1・2 | **詰まったら使う** ― まず 3 分間は自力で考える |
| 演習 問3（考察） | コードは OK、**考察文は自分で書く** |
| 演習 問4（発展） | **自由に活用** ― プロンプトの工夫も練習 |

> 💡 **Copilot を使う前に「何をしたいか」を日本語で言えるようにしましょう。**  
> 「何をしたいか」が言えないと良いプロンプトが書けません。それがわかれば Copilot が助けてくれます。

---
## 🔧 準備

### ライブラリを読み込む

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib
except ImportError:
    pass
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("✅ ライブラリ読み込み完了")

### データを読み込む

In [ ]:
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種"] = wine.target
species = {0: "Barolo", 1: "Grignolino", 2: "Barbera"}
df["品種名"] = df["品種"].map(species)

print(f"✅ データ準備完了  {len(df)}件・{df.shape[1]}列")

### 特徴量と正解ラベルを分ける

In [ ]:
X = df.drop(columns=["品種", "品種名"])   # 13列の特徴量
y = df["品種"]                             # 正解ラベル（0/1/2）

print(f"特徴量 X の形: {X.shape}  ← {X.shape[0]}件 × {X.shape[1]}変数")
print(f"正解ラベル y の種類: {sorted(y.unique())}  ← 0=Barolo, 1=Grignolino, 2=Barbera")

---
## 問1：訓練データとテストデータに分割する ★☆☆

**実務での対応：** モデルを本番環境に適用する前に、「まだ見ていないデータで精度を確認する」ステップです。

### なぜデータを分割するのか（復習）
「練習問題と同じ問題でテストしても実力は測れない」のと同じです。

```
全データ（178件）
     ↓ train_test_split で分割
訓練データ（80% ≈ 142件）← モデルが「学習」する
テストデータ（20% ≈ 36件）← 評価専用。学習には絶対に使わない
```

### 引数の意味
| 引数 | 意味 |
|------|------|
| `test_size=0.2` | 20% をテスト用に確保 |
| `random_state=42` | シード値。同じ数字なら毎回同じ分割になる（再現性の確保） |
| `stratify=y` | 各品種の割合を訓練・テストで同じに保つ（重要） |

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"訓練データ: {X_train.shape[0]}件  テストデータ: {X_test.shape[0]}件")
print(f"合計: {X_train.shape[0] + X_test.shape[0]}件  ← 元の 178 件と一致？")

### 品種の分布が訓練・テストで同じか確認する（stratify の効果）

In [ ]:
import pandas as pd

print("訓練データの品種割合：")
print((pd.Series(y_train).map(species).value_counts() / len(y_train)).round(3))

print("\nテストデータの品種割合：")
print((pd.Series(y_test).map(species).value_counts() / len(y_test)).round(3))

#### ✅ チェックポイント
- 訓練: 約142件、テスト: 約36件 になっていますか？
- 訓練とテストで品種の割合がほぼ同じになっていますか？（stratify の効果）

---
## 問2：スケーリング → 学習 → 予測 → 正解率 ★★☆

**実務での対応：** 「売上予測モデル」「顧客離脱予測モデル」を作る際の標準的なワークフローです。

### STEP 1：スケーリング ― 変数間の単位を揃える

**なぜ必要か：**
- `alcohol`：11〜15（範囲 ≈ 4）
- `proline`：278〜1680（範囲 ≈ 1400）

単位をそのまま使うと `proline` の影響が 350 倍大きくなってしまいます。

### 🔑 ルール①：データリーク防止
```python
# ❌ NG：テストに fit_transform はデータリーク
scaler.fit_transform(X_test)

# ✅ OK：テストには transform のみ
scaler.fit_transform(X_train)   # 訓練で学習して変換
scaler.transform(X_test)        # テストは変換のみ
```

In [ ]:
scaler = StandardScaler()

# 訓練データ：平均・標準偏差を「学習」して変換する
X_train_scaled = scaler.fit_transform(X_train)

# テストデータ：訓練データで学習した値を使って変換するだけ（fit しない！）
X_test_scaled  = scaler.transform(X_test)

print(f"スケーリング後の訓練データ 先頭3行・先頭3列：")
print(X_train_scaled[:3, :3].round(2))
print("→ 値が −2〜+2 程度の範囲になっていれば正常")

### STEP 2：ランダムフォレストで学習する

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 訓練データだけで学習する（テストデータは使わない）
model.fit(X_train_scaled, y_train)

print("✅ 学習完了")

### STEP 3：テストデータで予測する

In [ ]:
y_pred = model.predict(X_test_scaled)

print(f"予測結果（先頭10件）: {y_pred[:10]}")
print(f"正解ラベル（先頭10件）: {list(y_test[:10])}")

### STEP 4：正解率を計算する

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"✅ 正解率: {acc:.1%}  ({int(acc*len(y_test))}/{len(y_test)} 件正解）")
print()
print("⚠️  でも正解率だけでは不十分です。次の問3で「どう間違えたか」を確認します。")

#### ✅ チェックポイント
- 正解率が表示されましたか？何%でしたか？
- ランダムに推測した場合の正解率は約 33%（1/3）です。それより大幅に高いですか？

> 🤖 **Copilot を使ってみよう**
>
> `n_estimators`（決定木の本数）を変えると結果はどう変わるか確認してみましょう：
> ```
> # n_estimatorsを10, 50, 200に変えてそれぞれの正解率を比較するコード
> ```

---
## 問3：混同行列で「どう間違えたか」を確認する ★★☆（最重要）

**実務での対応：** 経営層に「このモデルはどんなミスをしますか？」と聞かれたときの回答資料です。

### 🔑 ルール②：正解率だけで評価しない理由（再確認）

**極端な例：**
- 1000件中990件が「正常」・10件が「異常」のデータで
- 「全部正常」と予測するモデル → 正解率99%（でも異常を1件も検知できない）

**混同行列で確認できること：**
```
              予測: 0    1    2
実際: 0  →  [  ?    ?    ?  ]  ← 行：実際の品種
実際: 1  →  [  ?    ?    ?  ]  ← 列：モデルの予測
実際: 2  →  [  ?    ?    ?  ]

対角成分（左上〜右下）= 正解した件数
対角以外              = 間違えた件数（どの品種と混同したか）
```

### 混同行列を計算して表示する

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=wine.target_names)
disp.plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"混同行列  （正解率: {acc:.1%}）", fontsize=13)
plt.tight_layout()
plt.show()

### 数値で詳細を確認する

In [ ]:
print("=== 混同行列（数値）===")
print(f"{'':15} {'予測:Barolo':>15} {'予測:Grignolino':>18} {'予測:Barbera':>15}")
for i, name in enumerate(wine.target_names):
    row = cm[i]
    print(f"実際:{name:12} {row[0]:>15}  {row[1]:>18}  {row[2]:>15}")

print()
print(f"正解件数合計: {cm.diagonal().sum()} / {cm.sum()} 件")
print(f"誤分類件数: {cm.sum() - cm.diagonal().sum()} 件")

### 考察欄

**最も誤分類されやすい品種はどれですか？**

> （記入してください）

**なぜその品種が混同されやすいと思いますか？**  
（Ch.2 の散布図・仮説を思い出してください）

> （例：Grignolino と Barbera は Ch.2 の散布図で分布が重なっていたので、  
>  モデルも区別しにくかった）

**実務でこのモデルを使うとしたら、どの誤分類が最も問題になりますか？**

> （例：高級品種 Barolo を安価品種と誤分類すると損失が大きい → この誤分類を0にするよう改善が必要）

---
## 問4（発展）：特徴量重要度で Ch.2 の仮説を検証する ★★★

### 特徴量重要度とは
ランダムフォレストは「どの変数が予測に貢献したか」を数値で出力できます。  
これを **特徴量重要度（Feature Importance）** と呼びます。

**Ch.2 で立てた仮説との答え合わせができます。**

In [ ]:
# 特徴量重要度を取り出して整形する
importance_df = pd.DataFrame({
    "特徴量": wine.feature_names,
    "重要度": model.feature_importances_
}).sort_values("重要度", ascending=False).reset_index(drop=True)

importance_df.head(5)

In [ ]:
# 棒グラフで重要度を可視化する
colors = ["#1E2E4A" if i == 0 else ("#E86A2C" if i < 3 else "#7A9CC0")
          for i in range(len(importance_df))]

plt.figure(figsize=(8, 6))
plt.barh(importance_df["特徴量"], importance_df["重要度"], color=colors)
plt.xlabel("重要度（高いほど予測に貢献している）")
plt.title("特徴量重要度ランキング（Random Forest）")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"最重要変数: {importance_df.iloc[0]['特徴量']}  重要度: {importance_df.iloc[0]['重要度']:.3f}")

### Ch.2 仮説との答え合わせ

**Ch.2 で立てた仮説：**

> （Ch.2 の問3に書いた仮説を確認してください）

**仮説と特徴量重要度の一致度：**

> 一致した / 外れた / 一部一致

**外れた場合、なぜ外れたと思いますか？**

> （記入してください）

> 🤖 **Copilot チャレンジ**
>
> 「上位3変数だけを使ってモデルを再学習したら正解率はどうなるか」を試してみましょう：
> ```
> # 特徴量重要度の上位3変数だけでRandomForestを学習して正解率を比較するコード
> ```
> 実務では「精度はほぼ変わらず、変数を大幅に削減できる」ことがよくあります。